# Ch5: Resampling Methods

### **Q5.** 

In [36]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from ISLP import confusion_table, load_data
from ISLP.models import ModelSpec as MS
from ISLP.models import summarize
from sklearn.model_selection import train_test_split

sns.set_theme()

%matplotlib inline

In [37]:
from ISLP.models import sklearn_sm
from sklearn.model_selection import KFold, cross_validate

### Load / Simulate the Default Data Set

The real ISLP Default dataset has 10,000 observations with columns: `default` (Yes/No), `student` (Yes/No), `balance`, `income`. We simulate a close replica here.

In [38]:
default = load_data('Default')
default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


In [39]:
default.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   default  10000 non-null  category
 1   student  10000 non-null  category
 2   balance  10000 non-null  float64 
 3   income   10000 non-null  float64 
dtypes: category(2), float64(2)
memory usage: 176.1 KB


In [40]:
np.unique(default['default'], return_counts=True)

(array(['No', 'Yes'], dtype=object), array([9667,  333], dtype=int64))

## (a) Fit a logistic regression model using income and balance to predict default

In [41]:
design = MS(['income', 'balance'])

X = design.fit_transform(default)
y = default['default'] == 'Yes'

glm = sm.GLM(y, 
            X,
            family=sm.families.Binomial())

results = glm.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-11.540500,0.435000,-26.544,0.0
income,0.000021,0.000005,4.174,0.0
balance,0.005600,0.000000,24.835,0.0


## (b) Validation set approach — single split

Steps:
1. Split the sample set into a training set and a validation set.
2. Fit a multiple logistic regression model using only the training observations.
3. Predict default status on the validation set (classify as "Yes" if posterior probability > 0.5).
4. Compute the validation set error (fraction misclassified).

In [44]:
design = MS(['income', 'balance'])

X = design.fit_transform(default)
y = default['default'] == 'Yes'

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=1)
lr = sm.GLM(y_train, 
            X_train,
            family=sm.families.Binomial())

results = lr.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-11.858100,0.528000,-22.477,0.0
income,0.000022,0.000006,3.707,0.0
balance,0.005900,0.000000,21.078,0.0


In [45]:
pred_proba = results.predict(X_valid)
pred_proba.head()

9953    0.000919
3850    0.008410
4962    0.000806
3886    0.003090
5437    0.092877
dtype: float64

In [46]:
pred = np.where(pred_proba > 0.5, 1, 0)
np.unique(pred, return_counts=True)

(array([0, 1]), array([2951,   49], dtype=int64))

In [47]:
conf_mat = confusion_table(pred, y_valid)
conf_mat

Truth,False,True
Predicted,,
False,2893,58
True,16,33


In [48]:
(58+ 16)/conf_mat.sum().sum()

0.024666666666666667

We can see that the validation set error is $2.47\%$.

## (c) Repeat with three different splits

In [51]:
# helper function that fits the same model above but takes a split_random_state to make a different split of the data
# returns the validation set error
def fit_and_test(split_random_state):

    design = MS(['income', 'balance'])

    X = design.fit_transform(default)
    y = default['default'] == 'Yes'

    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=split_random_state)
    results = sm.GLM(y_train, 
                X_train,
                family=sm.families.Binomial()).fit()
    
    pred_proba = results.predict(X_valid)
    pred = np.where(pred_proba > 0.5, 1, 0)
    conf_mat = confusion_table(pred, y_valid)
    return (conf_mat.iloc[0, 1]+ conf_mat.iloc[1, 0])/conf_mat.sum().sum()

In [52]:
fit_and_test(1)

0.024666666666666667

This is the result using the same split used in **(b)**.

In [53]:
fit_and_test(2)

0.023666666666666666

In [54]:
fit_and_test(3)

0.025

In [55]:
fit_and_test(4)

0.025333333333333333

**Comment:** The validation set error varies across different splits. This illustrates the inherent variability of the validation set approach — the estimated test error depends on which observations end up in training vs. validation. Despite this variability, the errors are all in a similar range.

## (d) Including the student dummy variable

In [57]:
design = MS(['income', 'balance', 'student'])

X = design.fit_transform(default)
y = default['default'] == 'Yes'

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=1)
lr = sm.GLM(y_train, 
            X_train,
            family=sm.families.Binomial())

results = lr.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-11.142800,0.59400,-18.743,0.000
income,0.000004,0.00001,0.367,0.713
balance,0.005900,0.00000,21.022,0.000
student[Yes],-0.683600,0.28000,-2.440,0.015


In [59]:
(conf_mat.iloc[0, 1]+ conf_mat.iloc[1, 0])/conf_mat.sum().sum()

0.024333333333333332

In [58]:
pred_proba = results.predict(X_valid)
pred = np.where(pred_proba > 0.5, 1, 0)
conf_mat = confusion_table(pred, y_valid)
conf_mat

Truth,False,True
Predicted,,
False,2895,59
True,14,32


**Comment:** Including the student dummy variable does not lead to a meaningful reduction in the test error rate. This is consistent with the findings in Chapter 4, where the student variable was not significant after accounting for balance and income.